# `notebook_6_v3_model` — основной инференс (метод C, score 0.44571)

Финальный пайплайн: Description-признаки + GMM-2 + 5% uncertain → кластер 2.  
Файл для Kaggle: **`submission3c.csv`** (или `submission3.csv` — копия 3c).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from sygnal_clustering.config import DATA_PATH
from sygnal_clustering.data import load_waveforms
from sygnal_clustering.pipeline_v3 import (
    SUBMISSION3_PATH,
    SUBMISSION3C_PATH,
    method_c_gmm2_low_confidence,
    labels_to_submission,
)

X = load_waveforms(DATA_PATH)
labels_c, _ = method_c_gmm2_low_confidence(X, uncertain_fraction=0.05)
from sygnal_clustering.config import SUBMISSION_PATH

labels_to_submission(labels_c, SUBMISSION3C_PATH)
labels_to_submission(labels_c, SUBMISSION3_PATH)
labels_to_submission(labels_c, SUBMISSION_PATH)
sub = pd.read_csv(SUBMISSION3C_PATH)
print(sub["cluster"].value_counts().sort_index())
display(sub.head())

EXP = ROOT / "Разработка" / "Эксперименты"
kaggle_final = {
    "rank": 29,
    "score": 0.44571,
    "submission": "submission3c.csv",
    "method": "method_c_gmm2_low_confidence",
    "uncertain_fraction": 0.05,
    "cluster_counts": sub["cluster"].value_counts().sort_index().to_dict(),
}
print(kaggle_final)

best_img = EXP / "kaggle_leaderboard_best_0.44571.png"
if best_img.exists():
    display(Image(filename=str(best_img)))
else:
    print("Скриншот не найден:", best_img)

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f'''### ML-архитектор-аналитик
**Итоговая модель:** `method_c_gmm2_low_confidence` (`pipeline_v3` + `signal_extraction`).
Kaggle: **{kaggle_final["score"]:.5f}**, место **#{kaggle_final["rank"]}** (лидерборд на скриншоте выше).
Распределение кластеров: {kaggle_final["cluster_counts"]} — доля класса 2 ≈5%, без коллапса ~90% в один класс (v1).
Почему победил C: бинарный GMM на Description-признаках + явный кластер неопределённости; v2 (баланс) и B (GMM-3) не улучшили accuracy.
**Рекомендация:** зафиксировать `submission3c.csv` как production; v1/v2 ноутбуки — в `Разработка/Эксперименты/`.

### Учёный-физик (радиация, ядерная энергетика; PhD predictive models)
Импульсы параметризованы по методичке (3σ от базовой линии, PSD, decay 40%). Классы 0/1 — две популяции γ/нейтроны по форме; класс 2 — ~5% событий с максимальной неопределённостью GMM (спорные/аномальные).
Score **{kaggle_final["score"]:.5f}** подтверждает физическую схему «два типа + хвост» лучше, чем чистый k=3 или метаданные col2.'''))